# Privacy Lens · 이미지 객체(번호판·얼굴) 검증·학습

D-16: YOLO 는 글자가 아닌 이미지 객체를 맡는다. 이 노트북은 **직접 학습 전에 공개 모델이 쓸 만한지** 재고, 필요하면 번호판을 학습한다.

| 칸 | 하는 일 | 시간(T4) |
|---|---|---|
| 0 → 1 | 설정·설치 | 3~5분 |
| 2 | 공개 번호판 모델(ONNX) 받기 + 해시 기록 | 1분 |
| 3 | Open Images 에서 번호판·얼굴 사진만 받아 YOLO 형식으로 | 5~15분 |
| 4 | 공개 번호판 모델 / YuNet 얼굴 검출을 **같은 Open Images 검증 사진**으로 재기 | 2~5분 |
| 5 | (선택) YOLO26n 으로 번호판 직접 학습(Open Images test 분할 사진) → 4번과 같은 검증 사진으로 비교 | 20~40분 |
| 6 | 결과를 드라이브에 저장 | 즉시 |
| 7 | 얼굴 다시 재기(묶음 라벨·작은 얼굴은 무시) + 점수 기준 바꿔 보기 | 5분 |
| 8 | 번호판 확신도 기준 바꿔 보기(공개 vs v001) | 3분 |
| 9 | 한국 번호판(Roboflow, CC BY 4.0)으로 재기 → v001 에서 이어 학습해 v002 | 30~40분 |
| 10 | 놓친 번호판 보기, 한국 번호판 합성으로 v003 | 60분 |

다음 학습(증명사진 모델·번호판 v004)은 `vision/colab_next.ipynb`. 과정 기록: `docs/OBJECTS_REPORT.md`.

런타임 → 런타임 유형 변경 → **T4 GPU**. 받는 사진은 Open Images(라벨 CC BY 4.0)이고 드라이브의 `PrivacyLens-YOLO/objects/` 에만 둔다(저장소 금지).
**주의:** Open Images 숫자는 해외 사진 기준이다. 한국 번호판·휴대폰 사진은 실사 평가 폴더로 따로 잰다.

## 0. 설정

In [ ]:
ROOT = "/content/drive/MyDrive/PrivacyLens-YOLO"   # 결과·모델을 둘 곳(공유 폴더 바로가기)
BRANCH = "feat/vision-objects"
PLATE_REPO = "ishaqinu/yolo26n-license-plate-detection"
PLATE_FILE = "alpr_26n.onnx"     # .pt 는 pickle 이라 열면 코드가 실행될 수 있다. ONNX 만 받는다
VAL_N = 300                      # 검증 사진 수(번호판·얼굴 각각)
TRAIN_N = 2000                   # 5번 칸 학습 사진 수
EPOCHS = 30

## 1. 드라이브 연결 · 코드 받기 · 설치

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, json, hashlib, pathlib
OUT = f"{ROOT}/objects"; os.makedirs(OUT, exist_ok=True)
if not os.path.isdir("/content/privacy-lens"):
    subprocess.run(["git", "clone", "-b", BRANCH, "https://github.com/2026-usw-project/privacy-lens", "/content/privacy-lens"], check=True)
%cd /content/privacy-lens
!git fetch -q origin && git checkout -q $BRANCH && git pull -q
!pip -q install -r requirements.txt "ultralytics>=8.4" onnxruntime-gpu huggingface_hub fiftyone
!python -m pipeline.objects download
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음(CPU)")

## 2. 공개 번호판 모델 받기

받은 파일의 커밋(revision)과 SHA-256 을 기록한다. 같은 파일인지 나중에 확인할 수 있게.

In [ ]:
from huggingface_hub import hf_hub_download, HfApi
rev = HfApi().model_info(PLATE_REPO).sha
plate_path = hf_hub_download(PLATE_REPO, PLATE_FILE, revision=rev)
digest = hashlib.sha256(open(plate_path, "rb").read()).hexdigest()
print("revision", rev, "\nsha256", digest, "\n", plate_path)
json.dump({"repo": PLATE_REPO, "file": PLATE_FILE, "revision": rev, "sha256": digest},
          open(f"{OUT}/plate_model_source.json", "w"), indent=1)

## 3. Open Images 에서 번호판·얼굴 사진만 받기

FiftyOne 으로 해당 클래스가 있는 사진만 받는다(전체 561GB 를 받지 않는다).

In [ ]:
import fiftyone as fo, fiftyone.zoo as foz
DATA = "/content/oi"

def export(cls, split, n, name):
    ds = foz.load_zoo_dataset("open-images-v7", split=split, label_types=["detections"], classes=[cls],
                              max_samples=n, only_matching=True, seed=0, shuffle=True,
                              dataset_name=f"{name}-{split}-{n}")
    ds.export(export_dir=f"{DATA}/{name}", dataset_type=fo.types.YOLOv5Dataset, label_field="ground_truth",
              split="val" if split == "validation" else "train", classes=[cls])
    n = len(ds)
    ds.delete()  # 메모리 아끼기
    return n

print("번호판 검증", export("Vehicle registration plate", "validation", VAL_N, "plate"))
print("얼굴 검증", export("Human face", "validation", VAL_N, "face"))

## 4. 공개 모델 재기 — 같은 Open Images 검증 사진

In [ ]:
from ultralytics import YOLO
import yaml
yp = f"{DATA}/plate/dataset.yaml"
cfg = yaml.safe_load(open(yp)); cfg["path"] = f"{DATA}/plate"
cfg.setdefault("train", cfg["val"])  # 검증만 받았을 때도 YAML 형식을 맞춘다(학습은 5번 칸)
yaml.safe_dump(cfg, open(yp, "w"))
r = YOLO(plate_path, task="detect").val(data=yp, split="val", imgsz=640, batch=1, verbose=False, plots=False)
plate_pub = {"mAP50": round(float(r.box.map50), 4), "mAP50_95": round(float(r.box.map), 4),
             "precision": round(float(r.box.mp), 4), "recall": round(float(r.box.mr), 4)}
print("공개 번호판 모델(Open Images 검증):", plate_pub)

# YuNet: 정답 얼굴(20px 이상)을 IoU 0.5 이상으로 맞힌 비율
from PIL import Image
from pipeline.objects import FaceDetector, _iou, MODELS, YUNET_FILE
from pipeline.types import Box
fd = FaceDetector(MODELS / YUNET_FILE)
tp = fn = fp = 0
for img in sorted(pathlib.Path(f"{DATA}/face/images/val").glob("*")):
    im = Image.open(img).convert("RGB"); W, H = im.size
    lab = pathlib.Path(str(img).replace("/images/", "/labels/")).with_suffix(".txt")
    gt = []
    for line in (lab.read_text().splitlines() if lab.exists() else []):
        _, cx, cy, w, h = map(float, line.split())
        b = Box(int((cx - w / 2) * W), int((cy - h / 2) * H), int(w * W), int(h * H))
        if b.w >= 20 and b.h >= 20:
            gt.append(b)
    pred = [o.box for o in fd.detect(im)]
    used = set()
    for g in gt:
        j = next((k for k, p in enumerate(pred) if k not in used and _iou(g, p) >= 0.5), None)
        if j is None: fn += 1
        else: tp += 1; used.add(j)
    fp += len(pred) - len(used)
face_pub = {"recall": round(tp / max(1, tp + fn), 4), "precision": round(tp / max(1, tp + fp), 4),
            "faces": tp + fn}
print("YuNet 얼굴(20px 이상):", face_pub)

## 5. (선택) 번호판 직접 학습 — YOLO26n

4번과 **같은 검증 사진**으로 잰다. 공개 모델보다 낫지 않으면 공개 모델을 쓴다.

In [ ]:
# Open Images 의 train 분할은 라벨 파일만 수 GB 라 무료 Colab 메모리(12GB)를 넘어 세션이 죽었다(2026-11-02).
# 그래서 test 분할(검증과 겹치지 않음)에서 학습 사진을 받는다.
print("번호판 학습", export("Vehicle registration plate", "test", TRAIN_N, "plate"))
cfg = yaml.safe_load(open(yp)); cfg["path"] = f"{DATA}/plate"
cfg.setdefault("train", cfg["val"])  # 검증만 받았을 때도 YAML 형식을 맞춘다(학습은 5번 칸)
yaml.safe_dump(cfg, open(yp, "w"))
m = YOLO("yolo26n.pt")
m.train(data=yp, epochs=EPOCHS, imgsz=640, batch=16, workers=2, project="/content/runs", name="plate", exist_ok=True,
        patience=10, verbose=False, plots=False)
best = "/content/runs/plate/weights/best.pt"
r = YOLO(best).val(data=yp, split="val", imgsz=640, verbose=False, plots=False)
plate_ours = {"mAP50": round(float(r.box.map50), 4), "mAP50_95": round(float(r.box.map), 4),
              "precision": round(float(r.box.mp), 4), "recall": round(float(r.box.mr), 4)}
print("우리 학습(같은 검증 사진):", plate_ours, "\n공개 모델:", plate_pub)
os.makedirs(f"{OUT}/plate_v001", exist_ok=True)
YOLO(best).export(format="onnx", imgsz=640)
!cp /content/runs/plate/weights/best.pt /content/runs/plate/weights/best.onnx "$OUT/plate_v001/"

## 6. 결과 저장

In [ ]:
import datetime
res = {"when": datetime.datetime.now().isoformat(timespec="minutes"), "val_images": VAL_N,
       "plate_public": plate_pub, "face_yunet": face_pub,
       "plate_ours": globals().get("plate_ours"), "train_images": TRAIN_N if "plate_ours" in globals() else 0}
json.dump(res, open(f"{OUT}/objects_eval.json", "w"), ensure_ascii=False, indent=1)
print(json.dumps(res, ensure_ascii=False, indent=1))

## 7. 얼굴 다시 재기 — 무시 규칙 + 기준값 바꿔 보기

4번 칸은 '여러 얼굴을 묶은 라벨(IsGroupOf)' 안의 얼굴과 20px 미만 얼굴을 찾은 것까지 오탐으로 셌다.
여기서는 그것을 무시로 따로 세고(`vision/objects_eval.py`), 점수 기준(지금 0.7)과 최소 크기를 바꿔 가며 잰다.
새 런타임이면 0·1번 칸만 먼저 돌리면 된다.

In [ ]:
import fiftyone as fo, fiftyone.zoo as foz, importlib
from PIL import Image
import pipeline.objects as po; importlib.reload(po)   # 앞 칸에서 옛 코드를 이미 불러왔을 수 있다
import vision.objects_eval as oe; importlib.reload(oe)
from pipeline.objects import FaceDetector, MODELS, YUNET_FILE
from pipeline.types import Box
FACE_N = 500
fds = foz.load_zoo_dataset("open-images-v7", split="validation", label_types=["detections"], classes=["Human face"],
                           max_samples=FACE_N, only_matching=True, seed=1, shuffle=True, dataset_name=f"face-attr-{FACE_N}")
fd_low = FaceDetector(MODELS / YUNET_FILE, score=0.3)   # 낮게 한 번 찾고, 기준은 나중에 바꿔 본다
face_samples = []
for s in fds:
    im = Image.open(s.filepath).convert("RGB"); W, H = im.size
    gts = []
    for d in (s.ground_truth.detections if s.ground_truth else []):
        x, y, w, h = d.bounding_box
        group = bool(d.has_field("IsGroupOf") and d["IsGroupOf"])
        gts.append(oe.GT(Box(int(x * W), int(y * H), int(w * W), int(h * H)), ignore=group))
    face_samples.append((gts, [oe.Pred(o.box, o.score) for o in fd_low.detect(im)]))
print("사진", len(face_samples), "얼굴 라벨", sum(len(g) for g, _ in face_samples),
      "묶음 라벨", sum(x.ignore for g, _ in face_samples for x in g))
THR = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
face_rows = []
for ms in (0, 20, 32):
    rows = oe.sweep(face_samples, THR, min_size=ms); face_rows += rows
    print(f"\n최소 크기 {ms}px"); print(oe.table(rows))
face_pick = oe.pick([r for r in face_rows if r["min_size"] == 20], min_precision=0.85)
print("\n고른 기준(20px 이상, 정밀도 0.85 이상 중 재현율 최대):", face_pick)

## 8. 번호판 기준값 바꿔 보기 — 공개 모델 vs 우리 v001 (Open Images 검증)

서비스는 확신도 0.4 이상만 쓴다(`PLATE_CONF`). 낮게 한 번 찾고 기준을 바꿔 가며 잰다.

In [ ]:
from ultralytics import YOLO
import importlib, vision.objects_eval as oe; importlib.reload(oe)   # 7번 칸을 건너뛰어도 되게
from PIL import Image
from pipeline.types import Box
def plate_preds(model_path, root, conf=0.05, imgsz=640):
    m = YOLO(str(model_path), task="detect"); out = []
    pairs = root if isinstance(root, list) else oe.yolo_pairs(pathlib.Path(root))
    for img, lab in pairs:
        im = Image.open(img).convert("RGB"); W, H = im.size
        r = m.predict(im, conf=conf, imgsz=imgsz, verbose=False)[0]
        preds = [oe.Pred(Box(int(a), int(b), int(c - a), int(d - b)), float(p))
                 for p, (a, b, c, d) in zip(r.boxes.conf.tolist(), r.boxes.xyxy.tolist())]
        out.append((oe.yolo_labels(lab, W, H), preds))
    return out
PTHR = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
V001 = f"{OUT}/plate_v001/best.onnx"
plate_models = {"public": plate_path, "v001": V001}
plate_rows = {}
for name, path in plate_models.items():
    rows = oe.sweep(plate_preds(path, f"{DATA}/plate/images/val"), PTHR)
    plate_rows[("oi", name)] = rows
    print(f"\n[{name}] Open Images 검증"); print(oe.table(rows))

## 9. 한국 번호판 — Roboflow 데이터로 재기·추가 학습

드라이브 `objects/kr/` 에 Roboflow 에서 **YOLOv8 또는 YOLO26 형식**(라벨 형식이 같다)으로 받은 zip 을 둔다(모두 CC BY 4.0). 파일 이름은 Roboflow 가 붙인 그대로 둬도 된다.
- `kr_eval.zip` — hyunjin/korea-car-license-plate(주차장 실사 207장). **평가에만** 쓴다
- `kr_train1.zip` — bowontron/license-plate-label, `kr_train2.zip` — korean-car-plat/korea-car-plat(가까이 찍은 번호판)

클래스 이름이 데이터마다 달라도 모두 '번호판' 하나로 합친다(아래에 클래스 이름을 찍어 확인한다).

In [ ]:
import zipfile, shutil, glob, yaml, datetime
KR = "/content/kr"
KR_ZIPS = {"kr_eval": "Korea Car License Plate*.zip", "kr_train1": "license-plate-label*.zip",
           "kr_train2": "korea car plat*.zip"}   # Roboflow 가 붙인 이름 그대로 둬도 된다
def unpack(name):
    found = sorted(glob.glob(f"{OUT}/kr/{name}.zip") + glob.glob(f"{OUT}/kr/{KR_ZIPS[name]}"))
    if not found: print("없음:", name, KR_ZIPS[name]); return None
    z = found[0]
    d = f"{KR}/{name}"; shutil.rmtree(d, ignore_errors=True); zipfile.ZipFile(z).extractall(d)
    y = glob.glob(f"{d}/**/data.yaml", recursive=True)
    names = yaml.safe_load(open(y[0])).get("names") if y else None
    n = len(oe.yolo_pairs(pathlib.Path(d)))
    for lab in glob.glob(f"{d}/**/labels/*.txt", recursive=True):   # 모든 클래스 → 0(번호판)
        lines = [l.split() for l in open(lab).read().splitlines() if l.strip()]
        open(lab, "w").write("".join("0 " + " ".join(p[1:]) + "\n" for p in lines))
    print(name, "사진", n, "원래 클래스", names); return d
kr_eval, kr_t1, kr_t2 = unpack("kr_eval"), unpack("kr_train1"), unpack("kr_train2")
if kr_eval:
    # Roboflow 는 학습 분할을 늘려(같은 사진 여러 장) 내보낸다. 원본 한 장씩만 평가한다
    kr_pairs = oe.roboflow_originals(oe.yolo_pairs(pathlib.Path(kr_eval)))
    for name, path in plate_models.items():
        rows = oe.sweep(plate_preds(path, kr_pairs), PTHR); plate_rows[("kr", name)] = rows
        print(f"\n[{name}] 한국 주차장 실사 {len(kr_pairs)}장"); print(oe.table(rows))

In [ ]:
# v001 에서 이어 학습: Open Images(5번 칸 학습 사진) + 한국 번호판 2종. 검증은 Open Images 검증 그대로
if not os.path.isdir(f"{DATA}/plate/images/train"):
    print("번호판 학습(Open Images)", export("Vehicle registration plate", "test", TRAIN_N, "plate"))
trains = [f"{DATA}/plate/images/train"] + [p for d in (kr_t1, kr_t2) if d
                                           for p in glob.glob(f"{d}/*/images") if "/test/" not in p + "/"]
cfg2 = {"path": "/", "train": trains, "val": f"{DATA}/plate/images/val", "names": {0: "plate"}}
yp2 = "/content/plate_kr.yaml"; yaml.safe_dump(cfg2, open(yp2, "w"), allow_unicode=True)
print(trains)
m = YOLO(f"{OUT}/plate_v001/best.pt")
m.train(data=yp2, epochs=20, imgsz=640, batch=16, workers=2, project="/content/runs", name="plate_kr", exist_ok=True,
        patience=0, verbose=False, plots=False)   # 조기 종료 끔(아래 설명)
# 검증이 Open Images 뿐이라 best.pt 는 해외 점수로 골라진다(2026-11-03: epoch 1 이 뽑혀 한국 재현율 0.51).
# 마지막 가중치(last.pt)가 한국 재현율 0.63 으로 훨씬 낫다. 그래서 last 를 쓴다
best2 = "/content/runs/plate_kr/weights/last.pt"
YOLO(best2).export(format="onnx", imgsz=640)
os.makedirs(f"{OUT}/plate_v002", exist_ok=True)
!cp /content/runs/plate_kr/weights/last.pt "$OUT/plate_v002/best.pt" && cp /content/runs/plate_kr/weights/last.onnx "$OUT/plate_v002/best.onnx"
plate_models["v002"] = f"{OUT}/plate_v002/best.onnx"
for where, root in (("oi", f"{DATA}/plate/images/val"), ("kr", kr_pairs if kr_eval else None)):
    if root:
        rows = oe.sweep(plate_preds(plate_models["v002"], root), PTHR); plate_rows[(where, "v002")] = rows
        print(f"\n[v002] {where}"); print(oe.table(rows))

In [ ]:
# 놓친 이유가 '작은 번호판'인지: 번호판 크기별 재현율, 그리고 사진을 크게(1280) 넣었을 때
if kr_eval:
    size_rows = {}
    for name, path in plate_models.items():
        for sz in (640, 1280):
            if sz == 1280 and path.endswith(".onnx"):
                continue  # ONNX 는 640 고정으로 내보냈다. 1280 은 .pt 로 잰다
            s = plate_preds(path, kr_pairs, imgsz=sz)
            size_rows[f"{name}@{sz}"] = {"sweep": oe.sweep(s, [0.3, 0.4, 0.5]), "by_size": oe.recall_by_size(s, thr=0.4)}
        pt = path.replace(".onnx", ".pt")
        if os.path.exists(pt):
            s = plate_preds(pt, kr_pairs, imgsz=1280)
            size_rows[f"{name}@1280"] = {"sweep": oe.sweep(s, [0.3, 0.4, 0.5]), "by_size": oe.recall_by_size(s, thr=0.4)}
    for k, v in size_rows.items():
        print(f"\n[{k}]"); print(oe.table(v["sweep"])); print("크기별 재현율(0.4):", v["by_size"])


In [ ]:
res2 = {"when": datetime.datetime.now().isoformat(timespec="minutes"),
        "face": ({"images": len(face_samples), "rows": face_rows, "pick": face_pick}
                 if "face_rows" in globals() else None),   # 7번 칸을 건너뛰었으면 비움
        "plate": {f"{w}:{n}": rows for (w, n), rows in plate_rows.items()},
        "kr_size": globals().get("size_rows"),
        "kr_sources": {"eval": "roboflow hyunjin/korea-car-license-plate (CC BY 4.0)",
                       "train": ["roboflow bowontron/license-plate-label (CC BY 4.0)",
                                 "roboflow korean-car-plat/korea-car-plat (CC BY 4.0)"]}}
name = "objects_eval_kr.json" if res2["face"] is None else "objects_eval2.json"   # 7번 결과를 덮어쓰지 않게
json.dump(res2, open(f"{OUT}/{name}", "w"), ensure_ascii=False, indent=1)
print("저장:", f"{OUT}/{name}")

## 10. 놓친 한국 번호판 보기 → 합성 사진으로 v003

v002 도 한국 주차장 번호판의 37% 를 놓친다(큰 번호판도). 한국 학습 사진이 '가까이 찍은 번호판'뿐이라서다.
`vision/plate_synth.py` 로 진짜 한국 번호판 조각을 Open Images 차량 장면의 번호판 자리에 붙여 '장면 + 한국 번호판' 사진을 만들고,
v002 에서 이어 학습한다. 검증에 한국 test 분할을 섞고, 조기 종료 없이 마지막 가중치를 쓴다. 평가 207장은 여전히 학습에 안 쓴다.
9번 칸까지 돌린 상태에서 실행한다(`kr_pairs`, `kr_t1`, `kr_t2` 필요).

In [ ]:
import importlib, vision.plate_synth as ps; importlib.reload(ps)
V002 = f"{OUT}/plate_v002/best.pt"
s_v002 = oe.predict_plates(V002, kr_pairs)
grid = ps.miss_grid(s_v002, kr_pairs, thr=0.4, limit=48)
grid.save(f"{OUT}/kr_missed_v002.jpg"); display(grid.resize((grid.width // 2, grid.height // 2)))

In [ ]:
crops = ps.collect_crops([pathlib.Path(kr_t1), pathlib.Path(kr_t2)])
n = ps.build(pathlib.Path(f"{DATA}/plate/images/train"), crops, pathlib.Path("/content/plate_synth"), keep=0.2)
print("번호판 조각", len(crops), "합성 사진", n)
v003 = ps.train_next(base=V002, oi_root=f"{DATA}/plate", synth_root="/content/plate_synth",
                     kr_dirs=[kr_t1, kr_t2], out_dir=f"{OUT}/plate_v003", epochs=20)
plate_models["v003"] = v003
res3 = {}
for nm in ("public", "v001", "v002", "v003"):
    p = plate_models.get(nm) or f"{OUT}/plate_{nm}/best.onnx"
    res3[nm] = {"kr": oe.sweep(oe.predict_plates(p, kr_pairs), [0.3, 0.4, 0.5]),
                "oi": oe.sweep(oe.predict_plates(p, oe.yolo_pairs(pathlib.Path(f"{DATA}/plate/images/val"))), [0.3, 0.4, 0.5])}
    print(f"\n[{nm}] 한국\n" + oe.table(res3[nm]["kr"]) + "\n[" + nm + "] Open Images\n" + oe.table(res3[nm]["oi"]))
json.dump(res3, open(f"{OUT}/objects_eval_v003.json", "w"), ensure_ascii=False, indent=1)